In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader

# Define transforms - COMPLETE THE MISSING PARTS
transform = transforms.Compose([
    transforms.Resize((28, 28)),                 # Resize to 28x28
    transforms.Grayscale(3),                     # Convert grayscale to RGB (Don't Touch!!)
    transforms.ToTensor(),                       # Convert to Tensor
    transforms.Normalize(                        # ImageNet normalization
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Load EMNIST letters dataset (given)
train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset = EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)

# Note: EMNIST letters has labels 1-26 (A-Z), so we have 26 classes
num_classes = 26

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")

In [ ]:
# Letter mapping (labels are 1-26 for A-Z)
letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

# Create DataLoaders and display samples
# Write your code here
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2
)

# Display one batch shape (sanity check)
images, labels = next(iter(train_loader))
print(images.shape, labels.min(), labels.max())


In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_v2_s

# Write your code here
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights
import torch.nn as nn

weights = EfficientNet_V2_S_Weights.IMAGENET1K_V1
model = efficientnet_v2_s(weights=weights)

# Freeze backbone
for p in model.features.parameters():
    p.requires_grad = False
# print(model)
# Replace classifier last Linear
in_f = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_f, 26)

# print(model)

In [ ]:
# Write your code here
import torch

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images = images.to(device)
        labels = (labels - 1).to(device)  # 1-26 -> 0-25

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images = images.to(device)
        labels = (labels - 1).to(device)  # 1-26 --------- > 0-25

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


In [ ]:
# Write your code here
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torchvision import models

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model (EfficientNetV2-S) + replace classifier for 26 classes
weights = models.EfficientNet_V2_S_Weights.IMAGENET1K_V1
model = models.efficientnet_v2_s(weights=weights)

# Freeze backbone (features)
for p in model.features.parameters():
    p.requires_grad = False

# Replace last layer
in_f = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_f, 26)

model = model.to(device)

# Loss + Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=1e-3)

# Train
epochs = 5
train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(epochs):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    va_loss, va_acc = validate(model, test_loader, criterion, device)

    train_losses.append(tr_loss)
    val_losses.append(va_loss)
    train_accs.append(tr_acc)
    val_accs.append(va_acc)

    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Train Loss: {tr_loss:.4f} | Train Acc: {tr_acc:.4f} "
          f"Val Loss: {va_loss:.4f} | Val Acc: {va_acc:.4f}")


In [ ]:
# Plot Loss
plt.figure()
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.legend()
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.show()

# Plot Accuracy
plt.figure()
plt.plot(train_accs, label="Train Acc")
plt.plot(val_accs, label="Val Acc")
plt.legend()
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.show()

In [ ]:
# Write your code here
import torch

@torch.no_grad()
def validate_tta(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images = images.to(device)
        labels = (labels - 1).to(device)  # 1-26 -> 0-25

        # Original
        out1 = model(images)

        # Horizontal flip (W dim = 3)
        h_flipped = torch.flip(images, dims=[3])
        out2 = model(h_flipped)

        # Vertical flip (H dim = 2)
        v_flipped = torch.flip(images, dims=[2])
        out3 = model(v_flipped)

        # Average logits
        outputs = (out1 + out2 + out3) / 3.0

        loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)

        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total

# Example usage:
tta_loss, tta_acc = validate_tta(model, test_loader, criterion, device)
print(f"TTA Val Loss: {tta_loss:.4f} | TTA Val Acc: {tta_acc:.4f}")
